> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t04_rerun_bronze_to_silver` · notebook `02_bronze_to_silver` · started 2026-10-11 13:02 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `mode=incremental`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 02 · Bronze → Silver
Casts epoch integers to timestamps, decodes titles, removes PII (drops names/avatars, hashes user ids with a
salted SHA-256), derives body features, applies data-quality rules and upserts every Silver table with
**`MERGE INTO`**. Re-running the same batches changes nothing (idempotent).

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: Bronze batches not yet in Silver | `backfill`: re-process the selected batches |
| `batch_ids` | empty | comma-separated batch ids |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `pii_salt` | empty (read from secret `so_lakehouse/pii_salt`) | same |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pii_salt", "")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.silver import run_bronze_to_silver

cfg = build_config(need_salt=True)
audit = run_bronze_to_silver(
    spark, cfg,
    mode=widget("mode"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    fail_on_error=True,
)

[SKIPPED] Bronze-to-Silver | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | no matching Bronze batches to process


In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
bb350a13-2112-4577-b4ff-84a5a47e7a02,bronze_to_silver-20261011T130221Z-e73c98,bronze_to_silver,Bronze-to-Silver,incremental,,*,,unprocessed batches,-,2026-10-11T13:02:23.190Z,2026-10-11T13:02:23.190Z,4.2e-05,SKIPPED,0,0,0,0,0,no matching Bronze batches to process,2026-10-11T13:02:23.190Z
